***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 5-Random walks on graphs and Markov chains  
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 7, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# IF RUNNING ON GOOGLE COLAB, UNCOMMENT THE FOLLOWING CODE CELL
# When prompted, upload: 
#     * mmids.py
#     * mathworld-adjacency.csv
#     * mathworld-titles.csv
# from your local file system
# Files at: https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main/utils
# Alternative instructions: https://colab.research.google.com/notebooks/io.ipynb

In [ ]:
#from google.colab import files
#
#uploaded = files.upload()
#
#for fn in uploaded.keys():
#    print('User uploaded file "{name}" with length {length} bytes'.format(
#      name=fn, length=len(uploaded[fn])))

In [ ]:
# FOR BOOK ONLY
import warnings
warnings.filterwarnings('ignore')
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
from numpy.random import default_rng
rng = default_rng(535)
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import tensorflow as tf
from tensorflow import keras
import mmids

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\P}{\mathbb{P}}$
$\newcommand{\E}{\mathbb{E}}$
$\newcommand{\S}{\mathcal{S}}$
$\newcommand{\indep}{\perp\!\!\!\perp}$   
$\newcommand{\bmu}{\boldsymbol{\mu}}$
$\newcommand{\bpi}{\boldsymbol{\pi}}$

## Background: review of conditioning

We first review the concept of conditioning, which generally plays a key role in probabilistic modeling and reasoning.

### Conditional probability

We start with events. Throughout, we work on a fixed probability space $(\Omega, \mathcal{F}, \P)$, which we assume is discrete, i.e., the number of elements in $\Omega$ is countable.

**DEFINITION** **(Conditional Probability)** Let $A$ and $B$ be two events with $\mathbb{P}[B] > 0$. The conditional probability of $A$ given $B$ is defined as

$$
\P[A|B] = \frac{\P[A \cap B]}{\P[B]}.
$$

$\natural$

The intuitive interpretation goes something like this: knowing that event $B$ has occurred, the updated probability of observing $A$ is the probability of its restriction to $B$ properly normalized to reflect that outcomes outside $B$ have updated probability $0$. 

<!--
This is illustrated next.

![Conditional probability](https://courses.cs.cornell.edu/cs2800/wiki/images/3/3b/Conditional-probability.svg)

([Source](https://courses.cs.cornell.edu/cs2800/wiki/index.php/File:Conditional-probability.svg))
-->

Conditional probabilities generally behave like unconditional probabilities.

Independence can be characterized in terms of conditional probability. In words, $A$ and $B$ are independent if conditioning on one of them having taken place does not change the probability of the other occurring. 

**LEMMA** **(Independence and Conditional Probability)** Let $A$ and $B$ be two events of positive probability. Then $A$ and $B$ are independent, which we will denote as $A \indep B$, if and only if $\P[A|B] = \P[A]$ and $\P[B|A] = \P[B]$. $\flat$

*Proof:* If $A$ and $B$ are independent, then c which implies

$$
\P[A|B] = \frac{\P[A \cap B]}{\P[B]} = \frac{\P[A] \P[B]}{\P[B]} = \P[A].
$$

In the other direction, 

$$
\P[A] = \P[A|B] = \frac{\P[A \cap B]}{\P[B]}
$$

implies $\P[A|B] = \frac{\P[A \cap B]}{\P[B]}$ after rearranging. $\square$

The conditional probability is often used in three fundamental ways, which we recall next. Proofs can be found in most probability textbooks.

- **Multiplication Rule:** For any collection of events $A_1,\ldots,A_r$,

$$
\P\left[\cap_{i=1}^r A_i\right]
= \prod_{i=1}^r \P\left[A_i \,\middle|\, \cap_{j=1}^{i-1} A_j \right].
$$


- **Law of Total Probability:** For any event $B$ and any [partition](https://en.wikipedia.org/wiki/Partition_of_a_set#Definition_and_Notation) $A_1,\ldots,A_r$ of $\Omega$,

$$
\P[B] 
= \sum_{i=1}^r \P[B|A_i] \P[A_i].
$$


- **Bayes' Rule:** For any events $A$ and $B$ with positive probability,

$$
\P[A|B]
= \frac{\P[B|A]\P[A]}{\P[B]}.
$$

It is implicit that all formulas above hold provided all conditional probabilities are well-defined.

### Conditioning on a random variable

Conditional probabilities extend naturally to random variables. If $X$ is a discrete random variable, we let $p_X$ be its probability mass function and $\S_X$ be its support, that is, the set of values where it has positive probability. Then we can for instance condition on the event $\{X = x\}$ for any $x \in \S_X$.

We define next the conditional probability mass function.

**DEFINITION** **(Conditional Probability Mass Function)** Let $X$ and $Y$ be discrete random variables with joint probability mass function $p_{X, Y}$ and marginals $p_X$ and $p_Y$. The conditional probability mass function of $X$ given $Y$ is defined as

$$
p_{X|Y}(x|y) := P[X=x|Y=y]  = \frac{p_{X,Y}(x,y)}{p_Y(y)}
$$

which is defined for all $x \in \S_X$ and $y \in \S_Y$. $\natural$

The conditional expectation can then be defined in a natural way as the expectation over the conditional probability mass function.

**DEFINITION** **(Conditional Expectation)** Let $X$ and $Y$ be discrete random variables where $X$ takes real values and has a finite mean. The conditional expectation of $X$ given $Y = y$ is given by

$$
\E[X|Y=y] = \sum_{x \in \S_X} x\, p_{X|Y}(x|y). 
$$

$\natural$

More generally, for a function $f$ over the range of $X$, we can define

$$
\E[f(X)|Y=y] = \sum_{x \in \S_X} f(x)\, p_{X|Y}(x|y). 
$$

We mention one useful formula: the *Law of Total Expectation*, the expectation version of the *Law of Total Probability*. It reads

$$
\E[f(X)] = \sum_{y \in \S_Y} \E[f(X)|Y=y] \,p_Y(y).
$$

### Conditional expectation as least-squares estimator

Thinking of $\E[X|Y=y]$ as a function of $y$ leads to a fundamental characterization of the conditional expectation.

**THEOREM** **(Conditional Expectation and Least Squares)** Let $X$ and $Y$ be discrete random variables where $X$ takes real values and has a finite variance. Then the conditional expectation $h(y) = \E[X|Y=y]$ minimizes the least squares criterion

$$
\min_{h} \E\left[(X - h(Y))^2\right]
$$

where the minimum is over all real-valued functions of $y$. $\sharp$

*Proof:* Think of $h(y)$ as a vector $\mathbf{h} = (h_y)_{y \in \S_Y}$, indexed by $\S_Y$ (which is countable by assumption), with $h_y = h(y) \in \mathbb{R}$. Then

\begin{align*}
\mathcal{L}(\mathbf{h})
&=\E\left[(X - h(Y))^2\right]\\
&= \sum_{x\in \S_X} \sum_{y \in \S_Y} (x - h_y)^2 p_{X,Y}(x,y)\\
&= \sum_{y \in \S_Y} \left[\sum_{x\in \S_X}  (x - h_y)^2 p_{X,Y}(x,y)\right].
\end{align*}


Expanding the sum in the square brackets (which we denote $q_y$ and think of as a function of $h_y$) gives

\begin{align*}
q_y(h_y)
&:= \sum_{x\in \S_X}  (x - h_y)^2 p_{X,Y}(x,y)\\
&= \sum_{x\in \S_X}  [x^2 - 2 x h_y + h_y^2] \,p_{X,Y}(x,y)\\
&= \left\{\sum_{x\in \S_X} x^2 p_{X,Y}(x,y)\right\}
+ \left\{- 2 \sum_{x\in \S_X} x p_{X,Y}(x,y)\right\} h_y
+ \left\{p_Y(y)\right\} h_y^2.
\end{align*}

By the *Miminizing a Quadratic Function Lemma*, the unique global minimum of $q_y(h_y)$ - provided $p_Y(y) > 0$ - is attained at

$$
h_y 
= - \frac{- 2 \sum_{x\in \S_X} x p_{X,Y}(x,y)}{2 p_Y(y)}.
$$

After rearranging, we get

$$
h_y 
= \sum_{x\in \S_X}  x \frac{p_{X,Y}(x,y)}{p_Y(y)}
= \sum_{x\in \S_X}  x p_{X|Y}(x|y)
= \E[X|Y=y]
$$

as claimed. $\square$